# Baseline Classifier Training — Clean vs. Dusty PV Panels
Trains a binary YOLOv8n-cls classifier on the Kaggle "Solar Panel DustDetection" dataset. Run in Google Colab (free T4 GPU).**This notebook is kept for documentation/reproducibility.** The loggedresult — **88.1% validation top-1 accuracy, 11 Aug 2026** — is the onecited in the paper. If you re-run this notebook, note that YOLO's trainingitself has stochastic elements beyond the `random.seed(42)` train/val split(data augmentation, weight initialization), so a fresh run will likely NOTreproduce exactly 88.3%. Don't overwrite `results_log.txt` with a newnumber unless you deliberately intend to redo this experiment.Colab's storage does not persist between sessions — every fresh run needsall setup cells re-run from scratch.

## 1. Download dataset
Kaggle credentials pulled from Colab Secrets — never uploaded directly as `kaggle.json`, never printed.

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
!kaggle datasets download -d hemanthsai7/solar-panel-dust-detection
!unzip -q solar-panel-dust-detection.zip -d /content/data

Dataset URL: https://www.kaggle.com/datasets/hemanthsai7/solar-panel-dust-detection
License(s): CC0-1.0
100% 1.18G/1.18G [00:11<00:00, 107MB/s]



## 2. Confirm dataset extracted correctly
Should print `['Clean', 'Dusty']`.

In [ ]:
import os; print(os.listdir('/content/data/Detect_solar_dust'))

['Clean', 'Dusty']


## 3. Install YOLOv8

In [ ]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.2 MB/s eta 0:00:00


## 4. Train/val split80/20 split, `random.seed(42)` for reproducibility.
Expected counts:train Clean 1194, train Dusty 855, val Clean 299, val Dusty 214.

In [ ]:
import os
import shutil
import random

source = "/content/data/Detect_solar_dust"
dest = "/content/data/Detect_solar_dust_split"

split_ratio = 0.8  # 80% train, 20% validation

random.seed(42)  # Makes the split reproducible if you rerun this

for class_name in ["Clean", "Dusty"]:
    images = os.listdir(os.path.join(source, class_name))

    random.shuffle(images)

    split_point = int(len(images) * split_ratio)

    train_imgs = images[:split_point]
    val_imgs = images[split_point:]

    for split_name, img_list in [
        ("train", train_imgs),
        ("val", val_imgs)
    ]:
        target_dir = os.path.join(dest, split_name, class_name)

        os.makedirs(target_dir, exist_ok=True)

        for img in img_list:
            shutil.copy(
                os.path.join(source, class_name, img),
                os.path.join(target_dir, img)
            )

print("Done. Train/val split created at:", dest)

Done. Train/val split created at: /content/data/Detect_solar_dust_split


## 5. Confirm split counts

In [ ]:
import os
for split in ["train", "val"]:
  for cls in ["Clean", "Dusty"]:
    path = f"/content/data/Detect_solar_dust_split/{split}/{cls}"
    print(split, cls, len(os.listdir(path)))

train Clean 1194
train Dusty 855
val Clean 299
val Dusty 214


## 6. Train the model`yolov8n-cls` — smallest/fastest variant, chosen for local hardware with no GPU.

20 epochs, 224px images.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n-cls.pt")
results = model.train(    data="/content/data/Detect_solar_dust_split",    epochs=20,    imgsz=224,    project="/content/runs",    name="baseline_clean_dusty")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/Detect_solar_dust_split, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, forma

## 7. Validate
Reports top-1 accuracy on the val split. Logged result: **88.1%**.

In [ ]:
metrics = model.val()
print(metrics.top1)  # top-1 accuracy

Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/data/Detect_solar_dust_split/train... found 2049 images in 2 classes ✅ 
val: /content/data/Detect_solar_dust_split/val... found 513 images in 2 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2665.2±864.9 MB/s, size: 1758.4 KB)
val: Scanning /content/data/Detect_solar_dust_split/val... 513 images, 0 corrupt: 100% ━━━━━━━━━━━━ 513/513 15.7Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 33/33 1.3it/s 25.6s
                   all      0.881          1
Speed: 0.0ms preprocess, 9.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val
0.8810915946960449


## 8. Download the trained weights
Downloads `best.pt` (the best-performing checkpoint, not `last.pt`).BUG FIX: previously hardcoded to a specific run-folder suffix(`baseline_clean_dusty-2`), which breaks if Ultralytics assigns adifferent suffix on a fresh run. Now finds the weights automatically.

In [ ]:
import glob, os
from google.colab import files
weight_paths = glob.glob('/content/runs/baseline_clean_dusty*/weights/best.pt')
weight_paths.sort(key=os.path.getmtime)
latest_weights = weight_paths[-1]
print("Using weights:", latest_weights)
files.download(latest_weights)

Using weights: /content/runs/baseline_clean_dusty/weights/best.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>